In [29]:
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages.utils import trim_messages, count_tokens_approximately



In [30]:
load_dotenv()

True

In [31]:
model = ChatOpenAI(model= 'gpt-4o-mini')

In [32]:
MAX_TOKENS = 300

In [33]:
def chat_node(state: MessagesState):

     # Trim conversation history -> last N messages that fit within the token budget
    messages = trim_messages(
        state["messages"],
        strategy="last",
        token_counter=count_tokens_approximately,
        max_tokens=MAX_TOKENS
    )

    print('Current Token Count ->', count_tokens_approximately(messages=messages))

    for message in messages:
        print(message.content)

    response = model.invoke(messages)

    return {"messages": [response]}

In [34]:
builder = StateGraph(MessagesState)
builder.add_node("chat_node", chat_node)
builder.add_edge(START, "chat_node")

In [35]:
checkpointer = InMemorySaver()

graph = builder.compile(checkpointer=checkpointer)

In [36]:
config = {"configurable": {"thread_id": "chat-1"}}

result = graph.invoke(
    {"messages": [{"role": "user", "content": "Hi, my name is Amitoj."}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 10
Hi, my name is Amitojs.


'Hi Amitojs! How can I assist you today?'

In [37]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "explain attention to me."}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 35
Hi, my name is Amitojs.
Hi Amitojs! How can I assist you today?
explain attention to me.


'Attention is a concept that originates from neuroscience and psychology, but it has also been adapted into various fields, including natural language processing (NLP) and machine learning. Here’s a breakdown of what attention generally means in different contexts:\n\n### General Definition\nIn a broad sense, attention refers to the process of selectively concentrating on a particular item or stimulus while ignoring others. It allows individuals or systems to focus resources on relevant information and be less affected by distractions.\n\n### In Psychology\nIn cognitive psychology, attention is viewed as a limited resource. Humans can only process a limited amount of information at one time, so attention helps filter out unnecessary data. This is crucial for tasks such as perception, memory, and decision-making.\n\n### In Neuroscience\nAttention is associated with certain neural mechanisms in the brain. Different regions, such as the prefrontal cortex and the parietal lobes, play roles

In [38]:
config = {"configurable": {"thread_id": "chat-1"}}

result = graph.invoke(
    {"messages": [{"role": "user", "content": "what is my name."}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 8
what is my name.


"I'm sorry, but I don't have access to personal information about users unless you provide it. If you'd like to share your name or any other details, feel free to do so!"